# 06 · Multi-Segment Trajectories

### Recap & why now
A real route has several waypoints, and the whole difficulty — and the whole payoff — is
at the joins.

Project 5 dodged the question by making the drone stop at every waypoint. Each segment
started and ended at rest, so the joins were trivially smooth and the flight was slow and
lurching. This notebook removes the stops: the drone will pass **through** its waypoints
at speed, and stay smooth because we ask for that in the constraints.

### Learning objectives
1. Write a multi-segment trajectory as one long vector of unknowns.
2. Express "hit this waypoint" and "be smooth at this join" as rows of a matrix.
3. Solve the whole route in one KKT system.
4. **Verify continuity** numerically rather than trusting the plot.
5. Build a 3-D trajectory by solving the same problem once per axis.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

## 1 · One long vector, local time

With $m$ segments and eight coefficients each, the unknown is one stacked vector of
$8m$ numbers, and the total cost is each segment's cost added up — a **block-diagonal**
$Q$, with nothing off the diagonal, because segment 1's coefficients never appear in
segment 2's snap.

Each segment keeps its own **local time**, running from 0 to its own duration. That
bookkeeping choice matters: if segment 5 started at global time 40 s we would be
computing $40^7 \approx 1.6 \times 10^{11}$ and the matrix would become badly conditioned.

In [ ]:
# === The multi-segment solver from Notebook 06 ===========================

def seg_row(N, seg, t, der):
    """A row of the big constraint matrix that touches only segment `seg`."""
    r = np.zeros(N)
    r[seg*NCOEF:(seg+1)*NCOEF] = deriv_row(NCOEF, t, der)
    return r

def build_cost(times, der=4):
    """Block-diagonal Q: one cost_matrix per segment, stacked along the diagonal."""
    Q = np.zeros((len(times)*NCOEF, len(times)*NCOEF))
    for s, T in enumerate(times):
        Q[s*NCOEF:(s+1)*NCOEF, s*NCOEF:(s+1)*NCOEF] = cost_matrix(NCOEF, T, der)
    return Q

def build_constraints(waypoints, times):
    """Waypoints, rest at both ends, and continuity of velocity/acceleration/jerk at each join."""
    m = len(times); N = m*NCOEF
    rows, vals = [], []
    for s in range(m):                             # Every segment starts and ends on its waypoints.
        rows.append(seg_row(N, s, 0.0, 0));      vals.append(waypoints[s])
        rows.append(seg_row(N, s, times[s], 0)); vals.append(waypoints[s+1])
    for der in (1, 2, 3):                          # At rest, in every sense, at both ends.
        rows.append(seg_row(N, 0, 0.0, der));          vals.append(0.0)
        rows.append(seg_row(N, m-1, times[m-1], der)); vals.append(0.0)
    for s in range(m - 1):                         # The two sides of each join must AGREE...
        for der in (1, 2, 3):
            rows.append(seg_row(N, s, times[s], der) - seg_row(N, s+1, 0.0, der))
            vals.append(0.0)                       # ...but we never say WHAT they agree on.
    return np.array(rows), np.array(vals)

def solve_min_snap_1d(waypoints, times, der=4):
    """Equality-constrained QP, solved through the KKT system. One axis."""
    Q = build_cost(times, der)
    A, b = build_constraints(waypoints, times)
    KKT = np.block([[2*Q, A.T], [A, np.zeros((len(b), len(b)))]])
    sol = np.linalg.solve(KKT, np.concatenate([np.zeros(Q.shape[0]), b]))
    return sol[:Q.shape[0]].reshape(len(times), NCOEF)      # Drop the Lagrange multipliers.

def sample(coeffs, times, t, der=0):
    """Evaluate the piecewise polynomial at global time t."""
    edges = np.concatenate([[0.0], np.cumsum(times)])
    if t <= 0:         return poly_val(coeffs[0], 0.0, der)
    if t >= edges[-1]: return poly_val(coeffs[-1], times[-1], der)
    s = int(np.searchsorted(edges, t, side="right") - 1)
    return poly_val(coeffs[s], t - edges[s], der)

def min_snap_3d(waypoints, times):
    """Solve each axis separately and wrap the result in the ref(t) interface the cascade wants."""
    W = np.asarray(waypoints, float)
    coeffs = [solve_min_snap_1d(W[:, axis], times) for axis in range(3)]
    total = float(np.sum(times))
    def ref(t):
        t = min(max(t, 0.0), total)
        p = np.array([sample(coeffs[a], times, t, 0) for a in range(3)])
        v = np.array([sample(coeffs[a], times, t, 1) for a in range(3)])
        acc = np.array([sample(coeffs[a], times, t, 2) for a in range(3)])
        if t >= total:
            v = np.zeros(3); acc = np.zeros(3)     # Hold position once the trajectory is finished.
        return p, v, acc
    return ref, total, coeffs

ROUTE = np.array([(0, 0, 0), (0, 0, 1.5), (2.0, 0, 1.5), (2.0, 2.0, 1.5), (2.0, 2.0, 2.5), (0, 0, 1.5)])
DURATIONS = [2.5, 3.0, 3.0, 2.0, 4.0]
g = 9.81
print("solver ready — the standing route has %d waypoints and %d segments, %.1f s total" %
      (len(ROUTE), len(DURATIONS), sum(DURATIONS)))

## 2 · Four kinds of row

Everything we want has the same shape — *(some derivative) at (some time) in (some
segment) equals (some number)* — so the whole specification is one linear system.

**Waypoints.** Each segment starts on its waypoint and ends on the next: $2m$ rows.
**Boundary conditions.** At rest at both ends of the whole route: 6 rows.
**Continuity.** At each interior join, segment $s$ and segment $s+1$ must agree on
velocity, acceleration and jerk. These rows have a **zero** right-hand side and touch two
segments at once.

Notice what continuity does *not* say: it never states what the velocity at a join
should be. It only says the two sides must agree, and the optimiser picks the value that
minimises snap. That single choice is what lets the drone carry speed through a waypoint.

In [ ]:
wps = [0.0, 2.0, 2.0, 0.0]                         # Out, hold position, come back.
times = [2.0, 1.5, 2.5]
A, b = build_constraints(wps, times)
m, N = len(times), len(times)*NCOEF
print("  unknowns    %d" % N)
print("  constraints %d  (= %d waypoint rows + 6 boundary + %d continuity)" %
      (len(b), 2*m, 3*(m-1)))
print("  free        %d  <- the room the optimiser works in" % (N - len(b)))

print("\na continuity row touches TWO segments — non-zeros on both halves:")
print(np.round(A[-1][:16], 3))
print("\nand its right-hand side is %.1f: 'agree', not 'be this value'." % b[-1])

## 3 · Solving the route

One `np.linalg.solve` on the whole thing. Then the check that matters: at every interior
join, do the two polynomials actually agree — in position, velocity, acceleration and
jerk — to machine precision?

In [ ]:
C = solve_min_snap_1d(wps, times)
edges = np.concatenate([[0.0], np.cumsum(times)])
print("waypoints hit exactly?", np.round([sample(C, times, t_) for t_ in edges], 9))

print("\ncontinuity at the interior joins:")
for s in range(len(times) - 1):
    for der, name in zip(range(4), ["position", "velocity", "accel   ", "jerk    "]):
        left = poly_val(C[s], times[s], der)        # End of one segment...
        right = poly_val(C[s+1], 0.0, der)          # ...and start of the next.
        print("   join %d  %s  %+9.5f  vs %+9.5f   diff %.1e" % (s+1, name, left, right, abs(left-right)))

print("\nvelocity AT the interior waypoints:", np.round([sample(C, times, t_, 1) for t_ in edges], 4))
print("Non-zero — the drone flies THROUGH those waypoints instead of stopping. Nobody told it")
print("what speed to carry; the optimiser chose the speeds that minimise snap.")

In [ ]:
tt = np.linspace(0, edges[-1], 800)
fig, axes = plt.subplots(1, 4, figsize=(14.5, 2.8))
for k, name in enumerate(["position", "velocity", "acceleration", "jerk"]):
    axes[k].plot(tt, [sample(C, times, t_, k) for t_ in tt], color="C0", lw=2)
    for e in edges[1:-1]:
        axes[k].axvline(e, color="C3", ls=":", lw=1.4)          # The joins.
    axes[k].set_title(name, fontsize=10); axes[k].set_xlabel("time [s]")
axes[0].plot(edges, wps, "o", color="C3", ms=7)
plt.tight_layout(); plt.show()

print("Red dotted lines are the segment joins. Every curve passes through them without a kink —")
print("that is the continuity constraints doing their job. Compare Project 5, where the velocity")
print("curve touched zero at every waypoint because we forced it to.")

## 4 · Three axes, three problems

The snap cost of a 3-D path is the sum of the snap costs of $x$, $y$ and $z$, and no
constraint written so far mixes the axes — so minimising the total means minimising each
axis on its own.

Solve the same problem three times, share the segment **times**, and you have a 3-D
trajectory. (Notebook 09's obstacle constraints *do* couple the axes, and that is exactly
when this convenience ends.)

In [ ]:
ref, T_total, coeffs = min_snap_3d(ROUTE, DURATIONS)
grid = np.linspace(0, T_total, 700)
P = np.array([ref(t_)[0] for t_ in grid])
V = np.array([ref(t_)[1] for t_ in grid])
A3 = np.array([ref(t_)[2] for t_ in grid])
print("duration %.1f s | peak speed %.2f m/s | peak acceleration %.2f m/s^2 | implied tilt %.1f°" %
      (T_total, np.linalg.norm(V, axis=1).max(), np.linalg.norm(A3, axis=1).max(),
       np.degrees(np.arctan(np.linalg.norm(A3[:, :2], axis=1).max()/g))))

fig = plt.figure(figsize=(12.5, 3.8))
ax = fig.add_subplot(121, projection="3d")
ax.plot(P[:, 0], P[:, 1], P[:, 2], color="C0", lw=2.4, label="optimised")
ax.plot(ROUTE[:, 0], ROUTE[:, 1], ROUTE[:, 2], color="0.7", ls="--", lw=1.4, label="straight route")
ax.plot(ROUTE[:, 0], ROUTE[:, 1], ROUTE[:, 2], "*", color="C3", ms=11)
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]"); ax.legend(fontsize=8)
ax.set_title("Through the waypoints"); ax.view_init(elev=24, azim=-62)
ax2 = fig.add_subplot(122)
for i, lbl in enumerate(["$v_x$", "$v_y$", "$v_z$"]):
    ax2.plot(grid, V[:, i], lw=1.7, label=lbl)
ax2.set_xlabel("time [s]"); ax2.set_ylabel("velocity [m/s]"); ax2.legend(fontsize=8)
ax2.set_title("Velocity — never returns to zero mid-flight")
plt.tight_layout(); plt.show()

print("The dashed grey line is the straight-line route. The optimised path ROUNDS the corners,")
print("because a rounded corner has less snap than a sharp one — while still passing exactly")
print("through every waypoint.")

## 🧪 Try it yourself

**E1.** The continuity rows have zero on the right-hand side. Why not simply *specify*
the velocity at each interior waypoint instead — what would that cost?

**E2.** Give the middle segment four times its duration and look at what the trajectory
does. Predict first: slower, or something else?

In [ ]:
# --- Solution E1 ---
def stop_at_every_waypoint(waypoints, times_):
    """Each segment solved on its own, at rest at both ends — Project 5's approach."""
    return np.vstack([solve_min_snap_1d([waypoints[s], waypoints[s+1]], [times_[s]])
                      for s in range(len(times_))])
C_free = solve_min_snap_1d(wps, times)
C_stop = stop_at_every_waypoint(wps, times)
Qfull = build_cost(times)
cost = lambda CC: float(CC.reshape(-1) @ Qfull @ CC.reshape(-1))
print("E1: we could specify them — but every value we invent is a constraint, and constraints")
print("    can only raise the minimum (Notebook 04). Leaving them free lets the optimiser choose.")
print("    flying through : snap cost %10.2f" % cost(C_free))
print("    forcing a stop : snap cost %10.2f  — %.0fx worse" % (cost(C_stop), cost(C_stop)/cost(C_free)))
print("    Stopping is not just slower, it is genuinely harder on the actuators: every stop is an")
print("    acceleration reversal, and reversals are what snap measures.")

# --- Solution E2 ---
print("\nE2:  middle duration   peak |v| per segment          middle segment reaches")
for mid in (1.5, 2.5, 4.0):
    tms = [2.0, mid, 2.5]
    Ck = solve_min_snap_1d(wps, tms)
    ed = np.concatenate([[0.0], np.cumsum(tms)])
    grid_ = np.linspace(0, ed[-1], 900)
    speeds = np.array([sample(Ck, tms, t_, 1) for t_ in grid_])
    per_seg = [np.abs(speeds[(grid_ >= ed[i]) & (grid_ <= ed[i+1])]).max() for i in range(3)]
    mid_grid = np.linspace(ed[1], ed[2], 400)
    print("    %13.1f s   %-28s %8.2f m" %
          (mid, np.round(per_seg, 2), max(sample(Ck, tms, t_) for t_ in mid_grid)))

print("    Everything speeds UP, and the middle segment most of all — the opposite of what 'more")
print("    time' suggests. The last column explains it: both ends of that middle segment sit at")
print("    2.0 m, so with four seconds to fill the drone does not wait. It swings out and comes")
print("    back, because standing still would mean decelerating to zero and re-accelerating, and")
print("    reversals are exactly what snap punishes.")
print("    This is the most important gotcha in minimum-snap planning: the optimiser was never")
print("    told to stay near the straight line. Notebook 09 fixes it properly with corridors.")

## 🚁 Mini-project: the route, flown as a reference

Animate a marker following the 3-D reference, with the straight-line route drawn behind
it. Watch it cut the corners smoothly while still touching every waypoint — and notice
that it never slows down in the middle.

In [ ]:
frames = np.linspace(0, T_total, 220)
speeds = np.array([np.linalg.norm(ref(t_)[1]) for t_ in frames])

fig = plt.figure(figsize=(9.8, 4.4))

def frame(k):
    fig.clf()
    ax = fig.add_subplot(121, projection="3d")
    ax.plot(ROUTE[:, 0], ROUTE[:, 1], ROUTE[:, 2], color="0.75", ls="--", lw=1.4)
    ax.plot(P[:, 0], P[:, 1], P[:, 2], color="0.88", lw=1.4)
    trail = np.array([ref(t_)[0] for t_ in frames[:k+1]])
    ax.plot(trail[:, 0], trail[:, 1], trail[:, 2], color="C0", lw=2.2)
    p_now = ref(frames[k])[0]
    ax.plot([p_now[0]], [p_now[1]], [p_now[2]], "o", color="C3", ms=9)
    ax.plot(ROUTE[:, 0], ROUTE[:, 1], ROUTE[:, 2], "*", color="C2", ms=9)
    ax.set_xlim(-0.6, 2.8); ax.set_ylim(-0.6, 2.8); ax.set_zlim(0, 3)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title("t = %5.2f s" % frames[k], fontsize=10); ax.view_init(elev=24, azim=-62)
    ax2 = fig.add_subplot(122)
    ax2.plot(frames[:k+1], speeds[:k+1], color="C1", lw=2)
    ax2.set_xlim(0, T_total); ax2.set_ylim(0, speeds.max()*1.15)
    ax2.set_xlabel("time [s]"); ax2.set_ylabel("speed [m/s]")
    ax2.set_title("speed never reaches zero mid-route", fontsize=10)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(frames), interval=45, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Everything in this notebook is the core of Mellinger and
> Kumar's minimum-snap generator, and it is one linear solve because the problem has only
> equality constraints. Production implementations add two things: they solve for the
> *joins'* derivative values directly, which is far better conditioned on long routes, and
> they optimise the segment times as well — which turns the problem non-convex and needs
> an outer loop. Notebook 08 does a simple version of the second.

**Where next.** Every requirement so far has been an equality. Real missions have
inequalities — stay under 3 m/s, stay inside this corridor — and the KKT trick cannot
express them. Notebook 07 brings in a solver that can.